# 🎯 Objetivo

### Visualizar los resultados de detección de anomalías de forma interactiva y clara, de modo que:

- Se puedan inspeccionar runs por pozo y etapa.
- Se vean los scores de anomalía en profundidad.
- Permita un primer paso hacia una app operativa o dashboard en producción.

### Visualización interactiva en Jupyter usando Plotly con dropdowns.

Enfocado en:

- Mostrar la señal CCL y su score de anomalía en profundidad.
- Permitir elegir un pozo y etapa con controles interactivos.
- Resaltar visualmente las zonas con alta probabilidad de anomalía.

### 💻 Celda 2 - Carga de datos con resultados de anomalía

In [ ]:
import pandas as pd

# Dataset procesado en etapa anterior
df = pd.read_csv("data/processed/ccl_anomaly_scores.csv")

# Vista rápida
df.head()

### 🧰 Celda 3 - Herramientas interactivas

In [ ]:
import plotly.graph_objs as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

### 📋 Celda 4 - Selección de pozo y etapa

In [ ]:
# Widgets para selección interactiva
pozos = sorted(df["pozo"].dropna().unique())
selector_pozo = widgets.Dropdown(options=pozos, description="Pozo")

def update_etapas(pozo_seleccionado):
    etapas = df[df["pozo"] == pozo_seleccionado]["etapa"].dropna().unique()
    return sorted(etapas)

selector_etapa = widgets.Dropdown(description="Etapa")

def on_pozo_change(change):
    selector_etapa.options = update_etapas(change['new'])

selector_pozo.observe(on_pozo_change, names='value')
on_pozo_change({'new': selector_pozo.value})

display(selector_pozo, selector_etapa)

### 📈 Celda 5 - Función para graficar run

In [ ]:
def plot_etapa(df, pozo, etapa):
    df_etapa = df[(df["pozo"] == pozo) & (df["etapa"] == etapa)].sort_values(by="DEPT")

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_etapa["CCL_norm"], 
        y=df_etapa["DEPT"], 
        mode='lines',
        name='CCL Normalizado',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        x=df_etapa["score_iso"], 
        y=df_etapa["DEPT"], 
        mode='lines',
        name='Score de Anomalía (Isolation Forest)',
        line=dict(color='red', dash='dot')
    ))

    fig.update_layout(
        title=f"Pozo: {pozo} | Etapa: {etapa}",
        xaxis_title="Valor",
        yaxis_title="Profundidad (DEPT)",
        yaxis_autorange='reversed',
        height=600
    )
    
    fig.show()

### 🖼️ Celda 6 - Ejecutar visualización

#### 💡 Nota: Podés volver a ejecutar la celda 6 con diferentes selecciones en los dropdowns para ver otros runs.

In [ ]:
plot_etapa(df, selector_pozo.value, selector_etapa.value)